# Lecture 05-07 - Vision and Grasping

# 1. 机器人抓取基础 (Fundamentals of Robotic Grasping)

抓取是具身智能中机器人与环境交互的基础能力。

### 1.1 定义与流程
*   **抓取 (Grasping):** 通过在一组接触点上施加力和力矩，以期望的方式限制物体运动的过程。
*   **抓取合成 (Grasp Synthesis):** 一个高维搜索或优化问题，旨在寻找合适的夹爪位姿或关节配置。
*   **标准流水线:**
    1.  **Grasp Synthesis:** 计算抓取位姿。
    2.  **Approaching:** 规划路径接近物体。
    3.  **Physical Interaction:** 物理接触与闭合。
    4.  **Lifting/Drop:** 操纵物体。

### 1.2 抓取控制回路 (Control Loop)
*   **开环抓取 (Open-Loop Grasping):** “看一眼，闭眼抓”。感知仅发生在动作开始前。
    *   *路径 A (已知物体):* 依赖 **6D姿态估计** 。
    *   *路径 B (未知/通用物体):* 直接预测抓取位姿 (End-to-End)。
*   **闭环抓取 (Closed-Loop Grasping):** 视觉伺服 (Visual Servoing)。在动作过程中持续跟踪物体位姿，实时修正误差。

### 1.3 抓取位姿的自由度 (DoF)
*   **4-DoF Grasp (Top-down):** $(x, y, z, \theta)$。通常指从上方垂直抓取，仅需3D位置和绕重力方向的旋转（偏航角）。
*   **6-DoF Grasp:** $(x, y, z, roll, pitch, yaw)$。全自由度抓取，包含3D位置和3D方向。

---

# 2. 6D 物体姿态估计 (6D Object Pose Estimation)

为了实现对已知物体的精准抓取，必须知道物体在相机坐标系下的精确状态。

### 2.1 定义
定义为从 **物体坐标系 (Object Space)** 到 **相机坐标系 (Camera Space)** 的刚体变换。包含 6 个自由度：
*   **3 DoF 平移 (Translation):** $T = [T_x, T_y, T_z]^T$
*   **3 DoF 旋转 (Rotation):** $R \in SO(3)$

数学表达为：
$$
\begin{bmatrix} X' \\ Y' \\ Z' \end{bmatrix}_{camera} = R \cdot \begin{bmatrix} X \\ Y \\ Z \end{bmatrix}_{object} + T
$$

注：只有在相机内参已知，物体实际大小已知的前提下，才能做到唯一确定物体位姿。

### 2.2 应用场景
*   **机器人操作:** 将CAD模型上的抓取点注释转移到相机空间。
*   **人机交互:** 视觉角度理解交互，机器人角度提取示教轨迹。
*   **增强现实 (AR):** 虚实融合渲染。

---

# 3. 核心数学问题：旋转表示 (Rotation Representation)

旋转的表示是深度学习进行姿态估计中的核心痛点。神经网络擅长输出欧几里得空间的值，而3D旋转位于非欧流形 $SO(3)$ 上。

### 3.1 常见的旋转表示及其缺陷
| 表示法 | 维度 | 定义/特点 | 缺陷 (Discontinuity/Singularity) |
| :--- | :--- | :--- | :--- |
| **旋转矩阵 (Rotation Matrix)** | $3 \times 3$ | $R^T R = I, \det(R)=1$ | 参数过多(9个)，正交约束难以在回归中保持。 |
| **欧拉角 (Euler Angle)** | 3D | 绕主轴旋转 ($\alpha, \beta, \gamma$) | **万向节死锁 (Gimbal Lock)**；表示不连续（周期性问题，如 $0$ 和 $2\pi$）。 |
| **轴角 (Axis-Angle)** | 3D | 向量 $v = \theta \mathbf{e}$ | 在 $\theta=0$ (单位阵) 和 $\theta=\pi$ 处存在不连续性/多义性。 |
| **四元数 (Quaternion)** | 4D | $q = w + xi + yj + zk$ | **双倍覆盖 (Double Coverage):** $q$ 和 $-q$ 表示同一旋转。若网络未约束半球，会导致不连续。 |

### 3.2 适用于神经网络的连续表示 (Continuous Representations)
为了解决上述不连续性导致网络难以收敛的问题，引入了高维连续表示：（如果ground truth不包含上面方法中的特殊突变位置，可以使用上面的表示方法而不产生问题）

*   **6D Representation (Zhou et al., CVPR 2019):**
    *   预测 $3 \times 2$ 的矩阵（旋转矩阵的前两列 $a_1, a_2$，第三列用前两列叉乘得到）。
    *   通过 **Gram-Schmidt 正交化** 恢复 $R$：
        $$
        b_1 = \text{Normalize}(a_1)
        $$
        $$
        b_2 = \text{Normalize}(a_2 - (b_1 \cdot a_2)b_1)
        $$
        $$
        b_3 = b_1 \times b_2
        $$
        $$
        R = [b_1 | b_2 | b_3]
        $$
    *   优势: 连续性好，易于神经网络学习。
    *   缺点：施密特正交化天然对于第一个向量更重视，第二个向量增加第一个向量方向的分量不会对结果产生改变。

*   **9D Representation (SVD-based):**
    *   直接回归 $3 \times 3$ 矩阵 $M$。
    *   通过 **SVD (奇异值分解)** 投影到 $SO(3)$：
        $$
        \hat{R} = U V^T \quad \text{where } M = U \Sigma V^T
        $$

---


# 4. 姿态估计的两种主流方法论

### 方法一：直接回归 (Direct Regression)
使用神经网络直接输出旋转和平移参数。

*   **典型案例:** **PoseCNN (RSS 2018)**
    *   **架构:** Encoder-Decoder 结构。
    *   **平移估计:** 预测物体中心点在图像上的像素位置 + 中心深度 $T_z$。
    *   **旋转估计:** 回归四元数 (Quaternion)。由于四元数的不连续性，通常效果不如几何方法精准。
    *   **处理遮挡与对称:** 利用语义分割和Hough Voting来定位中心。

### 方法二：基于对应关系的几何拟合 (Correspondence-based Fitting)
先预测几何对应关系，再通过数学优化求解位姿。这是目前精度较高的方法。

#### 1. 建立对应关系
*   网络不直接预测 $R, T$，而是预测图像中每个像素对应的 **3D物体坐标 (Object Coordinates)**。
*   从而建立 **2D-3D** (若无深度图) 或 **3D-3D** (若有深度图) 的稠密对应关系。

#### 2. 求解：正交普罗克汝斯特问题 (Orthogonal Procrustes Problem)
给定两组对应的3D点集 $M, N \in \mathbb{R}^{n \times 3}$，寻找最优旋转 $\hat{R}$ 使得：
$$
\hat{R} = \underset{R \in SO(3)}{\operatorname{argmin}} \| M^T - R N^T \|_F^2
$$

*   **解析解 (Analytical Solution):** 基于 SVD 分解。
    1.  计算协方差矩阵 $H = M^T N$。
    2.  进行 SVD 分解： $H = U D V^T$。
    3.  最优旋转 (考虑 $\det(R)=1$ 约束):
        $$
        \hat{R} = U \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & \det(U V^T) \end{bmatrix} V^T
        $$

*   **联合求解 R 和 T:**
    先去中心化 (Centering) 消除平移影响求解 $R$，再利用 $R$ 求解 $T$：
    $$
    \hat{T} = \bar{M}^T - \hat{R} \bar{N}^T
    $$

#### 3. 鲁棒估计：RANSAC
由于预测可能存在大量噪声（Outliers），必须使用 **RANSAC (Random Sample Consensus)** 算法：
1.  **随机采样:** 选取最小子集（例如3个点对）。
2.  **假设生成:** 计算当前的 $R, T$。
3.  **验证:** 计算所有点在当前模型下的内点 (Inliers) 数量。
4.  **迭代:** 重复上述过程，选择内点最多的模型，并用所有内点重新精细化拟合。

#### 4. PnP 算法 (Perspective-n-Point)
当只有 RGB 图像（2D-3D 对应关系）且已知相机内参 $K$ 时，使用 PnP 算法求解：
$$
s p_c = K [R|T] p_w
$$

---

# 5. 姿态估计的任务层级 (Task Scopes)

### 5.1 实例级 (Instance-Level)
*   **定义:** 针对特定的、已知的一小组物体进行估计。
*   **前提:** 训练和测试是同一个物体，且必须拥有该物体的精确 **CAD 模型**。
*   **数据集:** LINEMOD, YCB-Video.
*   **局限:** 无法泛化到未见过的新物体 (Novel objects)。

### 5.2 类别级 (Category-Level)
*   **定义:** 针对某一类物体（如“相机”、“杯子”）进行估计，即使该特定实例从未见过。
*   **核心挑战:** 类内形状差异 (Intra-category shape variations)。
*   **解决方案:** **NOCS (Normalized Object Coordinate Space)** (CVPR 2019 Oral)。
    *   将所有同类物体归一化到一个标准的 3D 空间中。
    *   RGB 颜色通道代表 XYZ 坐标。
    *   预测 Canonical Space 中的坐标，从而实现对未知实例的 6D 姿态和 3D 尺寸估计。

#### 核心公式 $P_{cam} = s \cdot R \cdot P_{nocs} + T$

这个公式描述的是**坐标系变换（Transformation）**。它将物体在“虚拟标准空间（NOCS）”中的每一个点，映射到“真实相机观测空间（Camera）”中的对应位置。

你可以把它看作是一个**“这就好比把一个标准乐高模型放到桌子上”**的过程。

1. 公式中的符号拆解

*   **$P_{nocs}$ (NOCS Point)**：
    *   **是什么**：这是物体在**归一化物体坐标系**中的坐标。
    *   **通俗理解**：想象你心里有一个完美的“标准相机模型”或者“标准杯子模型”。这个模型被压缩在一个边长为 1 的虚拟透明立方体（Unit Cube）里。
    *   不管现实中的杯子是大是小，在这个 NOCS 空间里，它们都被缩放进这个 $1 \times 1 \times 1$ 的盒子里，坐标范围通常是 $[-0.5, 0.5]$ 或 $[0, 1]$。
    *   **维度**：$3 \times 1$ 向量 $(x, y, z)$。

*   **$s$ (Scale, 尺度缩放)**：
    *   **是什么**：一个缩放系数（Scalar）。
    *   **为什么需要它**：这是**类别级**任务和**实例级**任务最大的区别。
        *   在**实例级**（如 PoseCNN）中，我们知道这就是特定的那个“可乐罐”，它的尺寸是固定的，所以不需要缩放（$s=1$）。
        *   在**类别级**中，我们只知道这是个“杯子”。但现实中，有的杯子高 10cm，有的高 20cm。
    *   **作用**：$s$ 把那个虚拟的、边长为 1 的“标准模型”放大到**真实世界的物理尺寸**（单位：米）。

*   **$R$ (Rotation, 旋转矩阵)**：
    *   **是什么**：一个 $3 \times 3$ 的旋转矩阵。
    *   **作用**：把放大后的物体，旋转到现实中它摆放的角度。

*   **$T$ (Translation, 平移向量)**：
    *   **是什么**：一个 $3 \times 1$ 的向量。
    *   **作用**：把放大、旋转后的物体，从原点搬运到相机看到的那个具体位置（比如桌子右上角，距离相机 1.5 米处）。

*   **$P_{cam}$ (Camera Space Point)**：
    *   **是什么**：这是物体表面那个点在**真实相机坐标系**下的坐标（单位：米）。
    *   **来源**：这个坐标通常是由**深度相机（Depth Camera）**直接测出来的。

NOCS 算法的核心逻辑是：
1. 网络预测 **$P_{nocs}$**（每个像素对应的归一化坐标）。
2. 深度相机告诉我们要拟合的目标 **$P_{cam}$**。
3. 我们利用 **Umeyama 算法**（一种解算点云变换的算法）解出这个方程中的未知数 **$s, R, T$**。这三个未知数就是我们要的 **6D 姿态 + 3D 尺寸**。

#### NOCS Map 为什么可以用颜色表示坐标？

你可能会觉得“坐标是数字，颜色是红绿蓝，这俩咋能是一回事？”。但在计算机视觉里，这是一种非常巧妙且常见的**可视化与数据表示**手段。

1. 数据的本质都是 3 个通道

请看两者的结构：
*   **3D 坐标 $(x, y, z)$**：由 3 个数字组成。在 NOCS 空间里，这些数字通常被归一化到 $0$ 到 $1$ 之间。
*   **RGB 颜色 $(r, g, b)$**：由 3 个数字组成。在计算机里，通常是 $0$ 到 $255$（整数）或者 $0.0$ 到 $1.0$（浮点数）。

2. 映射关系

我们可以直接建立一一映射：
*   **X 轴坐标 $\longleftrightarrow$ Red (R) 通道**
*   **Y 轴坐标 $\longleftrightarrow$ Green (G) 通道**
*   **Z 轴坐标 $\longleftrightarrow$ Blue (B) 通道**

假设 NOCS 空间的坐标范围是 $[0, 1]$：
*   物体**最左下后**的点 $(0, 0, 0)$ $\rightarrow$ 对应颜色 $(0, 0, 0)$ **黑色**。
*   物体**最右边**的点 $(1, 0, 0)$ $\rightarrow$ 对应颜色 $(1, 0, 0)$ **纯红色**。
*   物体**最上边**的点 $(0, 1, 0)$ $\rightarrow$ 对应颜色 $(0, 1, 0)$ **纯绿色**。
*   物体**中心**的点 $(0.5, 0.5, 0.5)$ $\rightarrow$ 对应颜色 $(0.5, 0.5, 0.5)$ **灰色**。

3. 为什么要这样“伪装”成图片？

**为了用上强大的 CNN（卷积神经网络）**：
CNN 非常擅长处理图片（预测像素的颜色）。如果我们告诉网络：“请你预测这张图里每个像素的‘颜色’”，网络会觉得很顺手。只不过我们定义这里的“颜色”其实代表的是“空间坐标”。
*   网络输出一张“彩色图”（NOCS Map）。
*   我们读出 R 通道的值，就知道这个像素点在物体坐标系的 X 轴位置。

**可视化调试（人类能看懂）**：
如果网络输出了三张灰度图分别代表 x,y,z，人类很难直观判断对错。
但如果合成一张 RGB 图（如 PPT 第 23, 26 页所示）：
*   你看那个相机模型的图，**左边是紫色的，右边是绿色的**，颜色过渡非常平滑。
*   这就意味着网络理解了物体的**几何连续性**。如果你看到预测出来的图颜色斑驳、杂乱，一眼就知道网络没训练好，因为它没有学到物体表面坐标是连续变化的。

# Lecture 07
## 力封闭 (Force Closure)
这是判断抓取是否成功的物理学金标准。
*   **定义**: 如果一组施加在物体上的摩擦接触力，其正向线性组合（Wrench Cones 的正跨度）能够覆盖整个 Wrench Space（即能抵抗任意方向的外力与力矩），则称该抓取为力封闭。
*   **关系**:
    $$ \text{Successful Grasp} \subseteq \text{Force Closure} \subseteq \text{Form Closure} $$
    *   **Form Closure (形封闭)**: 通过刚性接触完全限制物体运动（通常要求过多接触点，对二指夹爪过于严苛）。
    *   **Force Closure**: 引入摩擦力，是机器人抓取规划的常用最小要求。

## 数据集与数据合成 (Datasets & Synthesis)

深度学习驱动的抓取依赖于大规模数据。

### 2.1 数据获取流水线
1.  **合成数据 (Synthetic)**:
    *   **步骤**: 采样抓取候选点 $\rightarrow$ 物理模拟器评估 (Simulate) $\rightarrow$ 标记力封闭抓取为正样本。
    *   **代表**: **ShapeNet**, **ModelNet** (物体模型), **ACRONYM** (带抓取标注), **Objaverse-XL** (千万级3D物体)。
    *   **合成策略**: 堆叠 (Pile) 或 紧密排列 (Packed) 场景生成。
2.  **真实数据 (Real)**:
    *   **代表**: **GraspNet-1Billion**。
    *   **标注流程**: 点云采样抓取点 $\rightarrow$ 采样视角/平面内旋转/深度 $\rightarrow$ 投影到场景 $\rightarrow$ 碰撞检测 $\rightarrow$ 标注。

### 2.2 域随机化 (Domain Randomization)
为了解决 Sim-to-Real 的鸿沟，在合成数据（如 **DREDS** 数据集）时引入随机化：
*   **布局**: 物体实例、位置排列。
*   **材质**: BSDF 属性。
*   **环境**: 背景、光照、相机视角。

## 3. 视觉输入表征 (Visual Input Representation)

不同的数据结构决定了网络架构的选择。

| 表征方式 | 特点 | 对应网络 | 优缺点 |
| :--- | :--- | :--- | :--- |
| **Voxel Grids** | 3D网格，类似3D像素 | **VGN** | 显式几何，但受分辨率限制，计算量随体积立方增长。 |
| **Point Cloud** | 点集合，稀疏表示 | **GraspNet**, **GS-Net** | 显式几何，受传感器噪声和深度缺失影响（透明/反光物体）。 |
| **Images** | RGB 或 多视角 RGB | **MonoGraspNet**, **GraspNeRF** | 隐式几何，需推断深度或三维结构。 |


## 4. 抓取检测算法演进 (Grasp Detection Architectures)

### 4.1 基于体素的方法: VGN (Volumetric Grasping Network)
*   **核心思想**: 将场景体素化，直接预测每个体素的抓取质量。
*   **输入**: TSDF (Truncated Signed Distance Function) 体积。
*   **输出**: 三个体积张量。
    *   **Quality ($q$)**: 抓取成功率 $q \in [0, 1]$。
    *   **Orientation ($r$)**: 四元数表示。
    *   **Width ($w$)**: 夹爪开合宽度。
*   **后处理**: 高斯平滑 $\rightarrow$ 掩膜过滤 (Masking) $\rightarrow$ 非极大值抑制 (NMS)。
*   **评价**: 抗遮挡能力强（多视角融合），但受限于预定义的工作空间和分辨率。

### 4.2 基于点云的两阶段方法: GS-Net (Graspness Discovery)
针对复杂场景中的高精度抓取，引入了 **"Graspness" (可抓取度)** 概念。

*   **核心概念 Graspness**: 一种基于几何的质量度量，用于区分 cluttered 场景中的可抓取区域。
*   **架构**: **ResUNet14** (基于 MinkowskiEngine 的稀疏卷积 **SparseConv**)。
*   **流程 (Pipeline)**:
    1.  **Where to grasp (Graspness Discovery)**:
        *   输入点云 $N \times 3$。
        *   预测点级 Graspness 分数和 Objectness 分数。
        *   **FPS (最远点采样)**: 筛选出 $M$ 个 Seed Points (种子点)。
    2.  **How to grasp**:
        *   **View Selection**: 使用斐波那契格点 (Fibonacci lattices) 在球面上采样 $V$ 个接近方向。
        *   **Cylinder Grouping**: 以种子点为中心，在圆柱体内采样 $K$ 个邻域点提取特征。
        *   **Generation**: 通过 MLP 预测平面内旋转 (In-plane rotation) 和抓取深度。
*   **公式 (Graspness Measure)**:
    *   基于力分析模型计算，对于点 $i$，其视角级 Graspness $\tilde{s}_i^V$ 近似为在该视角下满足抓取质量阈值 $c$ 的候选抓取比例：
    $$ \tilde{s}_i^V = \left\{ \frac{\sum_{k=1}^{L} \mathbb{1}(q_k^{i,j} > c)}{|\mathcal{G}_{i,j}|} \mid 1 \leq j \leq V \right\} $$
    
#### 抓取全流程详解：分步细化与监督

GS-Net 利用局部特征，通过三个步骤把抓取姿态“找”出来。每个步骤都利用了**显式的监督信号（算 Loss）**，确保每一步都不走偏。

1. 第一步：Where to Grasp? (选点)
> **核心逻辑**：先在表面筛选出种子点。

*   **动作**：网络扫描整个点云，利用**局部特征**判断每一个点周围的几何形状是否适合下爪。
*   **监督 (Supervision)**：**Graspness Loss**。
    *   我们在训练时，提前用物理引擎算好每个点能不能抓。如果网络把一个光滑的大平面中心选为“好点”，Loss 就会惩罚它；如果它选中了一个把手上的点，Loss 就奖励它。
*   **结果**：剔除 90% 的无用区域，只保留高分种子点（Seed Points）。
  
1.  第二步：From Which Direction? (选方向)
> **核心逻辑**：在种子点对应半球的256个方向上确定爪子飞过来的角度。

*   **动作**：对于选中的种子点，网络在以它为中心的半球面上采样（比如 256 个 Fibonacci 格点方向），预测哪个方向进得去且抓得稳。
*   **监督 (Supervision)**：**View Selection Loss**。
    *   训练数据告诉网络：“对于这个把手点，垂直抓是错的（会撞），侧面抓是对的”。网络被迫学会辨别方向的好坏。
*   **结果**：为每个种子点锁定一个最佳的接近方向（View）。

1. 第三步：How to Pose? (定姿态)
> **核心逻辑**：圆柱体采样 + 最终参数回归。

*   **关键补充动作（圆柱体采样）**：确定了点和方向后，GS-Net 会沿着这个方向画一个**虚拟圆柱体**，把圆柱体内的**局部特征**提取出来。这一步非常关键，因为它保证了网络只看“手心里抓到的那部分形状”。
*   **动作**：根据这个圆柱体内的局部特征，网络最后预测三个东西：
    1.  爪子要伸多深（Depth）？
    2.  手腕要转多少度（In-plane Rotation）？
    3.  最后抓到的成功率是多少（Score/Quality）？
*   **监督 (Supervision)**：**Grasp Quality Loss & Classification Loss**。
    *   拿预测的参数去跟真值比对。如果预测的分数高但实际抓不住，或者预测的深度会撞到物体，都会产生 Loss。
*   **结果**：输出最终的 6-DoF 抓取姿态。


### 4.2.5 GS-Net 对 VGN 的改进
#### 1. 从“马赛克世界”到“高清世界”（精度的改进）

*   **VGN (体素法) —— 就像看“马赛克”画质：**
    *   VGN 把整个空间切成了一个个小方块（Voxel，体素），就像 Minecraft（我的世界）里的方块。
    *   **问题**：如果方块切得太大，抓取位置就不准（手可能抓空）；如果方块切得太小，计算量会爆炸，电脑跑不动。而且，它只能在预先设定好的一个长方体框框里抓，出了框就不认识了。
*   **GS-Net (点云法) —— 就像看“高清”矢量图：**
    *   GS-Net 直接处理**点云**（Point Cloud）。点云是物体表面实实在在的坐标点，是连续的，没有方块的限制。
    *   **改进**：它能精确地知道物体表面的曲率和细节，抓取的位置是“亚像素级”的精准，不再受限于方块的大小。

#### 2. 从“无脑遍历”到“有的放矢”（效率与策略的改进）

*   **VGN —— 盲目地“每一个格子都问一遍”：**
    *   VGN 不管面前是空气、桌子还是物体，它都会对空间里的每一个小方块进行计算：“这里能抓吗？那里能抓吗？”
    *   **问题**：空间里 90% 都是空气，只有一小部分是物体表面。VGN 把大量算力浪费在算空气上，效率低，且难以处理极其复杂的堆叠场景。
*   **GS-Net —— 聪明的“两步走”策略：**
    *   **第一步（Where）：** 先通过一个叫 **"Graspness" (可抓取度)** 的指标，快速扫描一遍点云，把那些明显不能抓的地方（比如桌子平面、物体内部、很难下手的角落）剔除掉，只保留“潜力股”（Seed Points，种子点）。
    *   **第二步（How）：** 针对这些“潜力股”，再精细地去算具体的抓取角度和宽度。
    *   **改进**：只在有意义的地方花算力，这就叫“有的放矢”，处理复杂杂乱场景（Clutter）的能力大大增强。

#### 3. 引入了“Graspness”这个新概念（核心导航仪）

VGN 只是单纯地预测“抓取质量（Quality）”，但这有时候不够用。

*   **VGN 的困境**：在杂乱的一堆物体中，可能有很多地方勉强能抓（Quality 还可以），但容易发生碰撞或者不太稳。
*   **GS-Net 的 Graspness**：这不仅是“能不能抓”，更像是一种**“几何上的抓取潜力热力图”**。
    *   GS-Net 会先看几何形状：“哎，这个把手凸出来了，很适合抓！”（Graspness 高）。
    *   “那个地方被压在最下面，虽然是物体的一部分，但根本伸不进去手。”（Graspness 低）。
    *   **改进**：它利用 **Graspness** 作为一个强有力的**先验知识（Prior）**，帮网络快速定位到最容易成功的区域，而不是在难抓的地方死磕。    

### 4.3 生成式与灵巧手抓取: DexGraspNet 2.0 & Diffusion
针对灵巧手 (Dexterous Hand) 的高维动作空间和多模态分布。

*   **挑战**: 灵巧手具有更强的包裹能力 (Wrapping ability)，但自由度极高，抓取分布是多模态的。
*   **模型**: **Conditional Grasp Generative Model (条件抓取生成模型)**。
*   **流程**:
    *   **Stage 1 (Where)**: 使用稀疏卷积提取局部特征 (Local Feature)，预测 Objectness 和 Graspness。
    *   **Stage 2 (How - Diffusion)**: 利用 **扩散模型 (Diffusion Model)** 处理多模态分布。
        *   条件生成: Condition 设为 Stage 1 提取的 Local Feature。
        *   预测目标: 腕部残差位置 $T$、旋转 $R$。手指关节角度 $\theta$ 则直接回归预测。
*   **Scaling Law**: 实验表明，增加抓取姿态数量和场景数量能显著提升性能。


## DexGraspNet 2.0 详解
---

### 1. 核心概念定义：

在灵巧手抓取中，我们需要确定两组主要的参数来完全描述一个抓取动作：

*   **$T$ (Translation，平移)**：
    *   **含义**：机器人手腕（Wrist/Base）在三维空间中的坐标位置 $(x, y, z)$。
    *   模型预测的通常是“残差位置”。即：$T_{final} = P_{seed} + T_{residual}$（种子点位置 + 预测的偏移量）。
*   **$R$ (Rotation，旋转)**：
    *   **含义**：机器人手腕的空间朝向。通常用 $3\times3$ 的旋转矩阵或四元数表示。它决定了手是从侧面抓、上面抓还是斜着抓。
    *   **$T$ 和 $R$ 统称为 6D Pose**：它们决定了**手掌**在哪里、朝向哪里。
*   **$\theta$ (Joint Angles，关节角)**：
    *   **含义**：灵巧手各个手指关节的弯曲角度。例如Shadow Hand有20多个自由度，$\theta$ 就是一个包含这20多个角度值的向量。
    *   **作用**：决定了**手指**的姿态（张开还是闭合）。

---

### 2. Stage 1: Where to grasp

*   **输入**：全场景点云。
*   **方法**：使用 **稀疏卷积 (Sparse Convolution)**。
*   **目的**：提取每个点的 **Local Feature (局部特征)**。
    *   *为什么要局部特征？* 泛化性强。一个杯子的把手，无论放在桌子上还是书架上，其局部几何形状是一样的。模型只需要认出“这是一个适合抓取的几何结构”，而不需要关心它在房间的绝对位置。
*   **输出**：
    *   **Objectness**：判断该点是否在物体表面（过滤掉背景噪声）。
    *   **Graspness**：判断该点是否适合作为抓取中心（Seed Point）。

---

### 3. Stage 2: How to grasp (核心难点解析)

这一步的目标是：给定一个提取好的 Local Feature（来自Stage 1），生成具体的抓取姿态 ($T, R, \theta$)。

#### 问题：为什么要区别对待 ($T, R$) 和 $\theta$？

**1. 针对 $T$ 和 $R$：多模态分布 (Multi-modal Distribution)**
*   **现象**：对于同一个抓取点（比如杯把手），机器人可以从**很多不同的方向**去抓。
*   **困难**：这些方案都是对的（即多峰分布/多模态）。如果我们用传统的监督学习（回归模型，如MSE Loss）去训练：
    *   模型会试图“取平均”。
    *   **结果**：这个平均出的位姿往往是**不合法**的。
*   **解决方案**：使用 **Diffusion Model**。Diffusion 天生擅长处理多模态分布，它不会取平均，而是能够**随机采样**生成出方案A、或者方案B、或者方案C中的某一个具体且合法的位姿。

**2. 针对 $\theta$：条件单峰分布 (Conditional Uni-modal)**
*   **现象**：一旦手腕的位置 ($T$) 和朝向 ($R$) **确定了**，手指该怎么弯曲其实就基本确定了。
    *   例如：手腕已经贴近了杯壁（$T, R$已定），手指为了抓紧，必须向内闭合接触物体表面。此时手指的角度变化范围很小，基本是确定的。
*   **逻辑**：$\theta$ 的分布在给定 $T, R$ 的条件下，退化为一个近似的单峰分布。
*   **解决方案**：直接使用 **MLP (多层感知机)** 进行回归预测。因为是单峰，简单的回归就能在这个特定的 $T, R$ 下算出最合理的手指角度。

---

### 4. Diffusion Model

#### 4.1 什么是 Diffusion Model
*   **前向过程 (加噪)**：把一张清晰的图（或一个合法的 $T, R$ 位姿）慢慢加上高斯噪声，直到它变成纯噪声。
*   **逆向过程 (去噪/生成)**：训练一个神经网络，让它学会**从纯噪声中一步步减去噪声**，还原出清晰的数据。
*   **生成能力**：当我们想生成一个新的抓取位姿时，我们随机采样一个纯高斯噪声，扔给网络，网络通过几十步的“去噪”，这就变出了一个合法的 $T, R$。

#### 4.2 在 DexGraspNet 中的具体流程 (Pipeline)

1.  **输入条件**：
    从 Stage 1 拿到某一个点的 **Local Feature**（我们称之为 $f$）。这个 $f$ 包含了这个点周围的几何信息（比如“这里是个圆柱面”）。

2.  **生成 $T$ 和 $R$ (Diffusion 过程)**：
    *   **初始化**：随机采样一个纯噪声向量 $\mathbf{x}_K$（代表完全混乱的位姿）。
    *   **去噪迭代**：将 $\mathbf{x}_K$ 和条件 $f$ 一起输入到 Diffusion 网络中。网络逐步预测噪声并去除，经过 $K$ 步后，得到清晰的、合理的 **手腕位姿 ($T, R$)**。
    *   *公式逻辑*：$p(T, R | \text{Local Feature})$。

3.  **生成 $\theta$ (回归过程)**：
    *   **输入**：
        1.  刚刚生成的 **$T, R$**。
        2.  之前的条件 **Local Feature $f$**。
    *   **计算**：通过一个简单的 MLP 网络（Joint MLP）。
    *   **输出**：预测出 **手指关节角度 $\theta$**。
    *   *公式逻辑*：$\theta = \text{MLP}(T, R, \text{Local Feature})$。

这就是 DexGraspNet 所谓的 **"End-to-End Generative Grasp Prediction Pipeline"**。

## 可供性与功能性操作 (Affordance & Manipulation)
### 1. Where2Act: From Pixels to Actions
**核心逻辑**：不仅仅是看物体，而是看“哪里可以交互”以及“能做什么动作”。

#### (1) 输入与输出
*   **Input**: 3D 物体的视觉信息（通常是 RGB-D 图像或点云 $P$）。
*   **Action Primitives**: 定义了基本的动作元语，主要是 **Push（推）** 和 **Pull（拉）**。
*   **Output (The Heatmap)**:
    *   网络输出的是一张**逐像素（Pixel-wise）或逐点（Point-wise）的评分图（Score Map）**。
    *   **热力图含义**：图上每一个像素的颜色深浅（或数值大小 $S \in [0, 1]$），代表了**“如果在这个点执行推/拉动作，成功的概率有多大”**。
        *   **红色/高分区域**：通常集中在门把手、抽屉边缘等可操作部位。
        *   **蓝色/低分区域**：墙面、柜子背面等不可移动的部位。

#### (2) 学习过程 (How it learns)
*   **数据来源**：**自监督交互 (Self-supervised Interaction)**。
    *   在仿真环境中，机器人随机尝试在物体的不同位置进行推或拉。
    *   如果物体发生了位移（比如门开了），记为**正样本 (Success, label=1)**。
    *   如果纹丝不动，记为**负样本 (Failure, label=0)**。
*   **训练目标**：
    *   训练一个网络（通常是 3D UNet 变体），输入几何信息，输出预测的分数 $S_{pred}$。
    *   利用交叉熵损失函数（Cross Entropy Loss）让 $S_{pred}$ 逼近真实的交互结果（0或1）。
*   **推理 (Inference)**：
    *   机器人拿到一张新物体的图 $\rightarrow$ 网络生成热力图 $\rightarrow$ 机器人选择热力图上**分数最高（Argmax）**的点作为接触点 $\rightarrow$ 执行动作。

---

### 2. VAT-Mart: Visual Action Trajectory
**核心逻辑**：光知道“点哪里（Where）”是不够的，开门是一个画圆弧的过程，开抽屉是一个走直线的过程。VAT-Mart 进一步生成了**轨迹（Trajectory）**。

#### (1) 进阶的热力图 (Interaction Heatmap)
*   VAT-Mart 依然保留了 Where2Act 的热力图机制，用于定位**接触点（Contact Point）**。
*   **改进点**：它不仅预测“能不能动”，还隐含了对**物体关节类型**的理解（是旋转轴 Revolute Joint 还是平移轴 Prismatic Joint）。

#### (2) 轨迹生成 (Trajectory Proposal)
除了热力图，VAT-Mart 还并行输出了**向量场（Vector Field）**或**轨迹参数**。
*   **Input**: 分割后的物体部件点云。
*   **Output**:
    1.  **Affordance Heatmap**: 哪里适合抓/推（同 Where2Act）。
    2.  **Trajectory Parameters**:
        *   对于**平移关节**（抽屉）：预测拉动的方向向量 $\mathbf{v}$。
        *   对于**旋转关节**（门/盖子）：预测旋转轴的位置、方向以及旋转半径 $r$。
*   **可视化理解**：
    *   Where2Act 的热力图告诉你：“请按这个红色的点”。
    *   VAT-Mart 的结果告诉你：“按住这个红色的点，并且**沿着这条蓝色的弧线**拉开”。

#### (3) 为什么叫 "VAT" (Visual Action Trajectory)?
*   因为很多操作动作不是瞬间完成的，而是一个时序的路径。
*   模型通过学习大量的交互数据，学会了将**视觉外观（Visual）**映射到**运动轨迹（Trajectory）**。例如，看到类似把手的长条状物体，模型就会生成一个“沿法线方向拉出”的直线轨迹。
